# 06 · Grid, random, Optuna, CV anidada y comparación honesta — Ames Housing

**Módulo 3 · Sesión 8** — Evaluación y selección de modelos

## Objetivos

Cierra el módulo reemplazando, con las herramientas de `05-sesgo-varianza-validacion.md` y
`06-seleccion-hiperparametros.md`, todo lo que las sesiones 6 y 7 dejaron pendiente:

1. Elegir $\lambda$ con **validación cruzada** en vez del split de validación único e
   informal que usó `04-regularizacion-aplicado.ipynb` (sesión 7).
2. Comparar grid search, random search y Optuna sobre un espacio de dos hiperparámetros, y
   **medir** cuánto le cuesta a cada uno llegar a un resultado igual de bueno.
3. Medir el sesgo optimista de reportar el error sobre el mismo CV que eligió el
   hiperparámetro, con **CV anidada**.
4. Formalizar la comparación pareada que `04-pipeline-caracteristicas-aplicado.ipynb`
   (módulo 2) hizo de manera informal, sobre Ridge vs. Lasso — y ver que la conclusión
   depende de si se respeta o no la separación del punto 3.
5. Cerrar con el único número que ninguna decisión de este notebook ha tocado: el RMSE
   sobre el conjunto de prueba.

**Paquetes:** `pandas`, `numpy`, `scikit-learn`, `scipy`, `optuna`.

In [ ]:
import time

import numpy as np
import pandas as pd
import optuna
from scipy.stats import loguniform, uniform
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    RandomizedSearchCV,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)
SEMILLA = 42

## 1. Mismos datos y pipeline de las sesiones 6 y 7

In [ ]:
datos = pd.read_csv("../datos/ames-housing.csv")
numericas = [
    "gr_liv_area", "total_bsmt_sf", "garage_area", "garage_cars",
    "overall_qual", "overall_cond", "year_built", "year_remod_add",
    "lot_area", "full_bath", "bedroom_abvgr",
]
categoricas = ["bldg_type", "house_style", "central_air"]

X = datos[numericas + categoricas]
y = datos["saleprice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA)

def crear_pipeline(regresor):
    """Pipeline nuevo, con preprocesador propio, para el regresor dado.

    `GridSearchCV` y `RandomizedSearchCV` clonan el estimador que reciben, así que con ellos
    daría igual; pero este notebook también ajusta pipelines **a mano** dentro de bucles
    (secciones 3 y 5), y ahí compartir un mismo `ColumnTransformer` haría que cada `fit`
    pisara el ajuste del anterior, sin lanzar ninguna excepción.
    """
    prep = ColumnTransformer(
        [
            ("num", Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())]), numericas),
            ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categoricas),
        ]
    )
    return Pipeline([("prep", prep), ("reg", regresor)])

## 2. Grid search con validación cruzada, reemplazando el split único de la sesión 7

El `kf` con semilla fija (`05-sesgo-varianza-validacion.md`, `06-seleccion-hiperparametros.md`
sección 4) se reutiliza en todas las búsquedas de este notebook, para que sean comparables
entre sí.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEMILLA)

pipe_ridge = crear_pipeline(Ridge())
grid_ridge = {"reg__alpha": np.logspace(-2, 3, 20)}

busqueda_ridge = GridSearchCV(pipe_ridge, grid_ridge, cv=kf, scoring="neg_root_mean_squared_error")
busqueda_ridge.fit(X_train, y_train)

print(f"Mejor alpha (Ridge): {busqueda_ridge.best_params_['reg__alpha']:.2f}")
print(f"RMSE de validación cruzada: ${-busqueda_ridge.best_score_:,.0f}")

El $\alpha$ elegido por 5-fold CV (≈7.8) es pequeño — coherente con la curva de validación
de la sesión 7, que ya mostraba el mínimo cerca de $\lambda \to 0$. La diferencia es que
ahora el número viene acompañado del respaldo de 5 pliegues, no de un solo split.

## 3. Grid vs. random vs. Optuna, con dos hiperparámetros

Con un solo hiperparámetro grid search no tiene gran desventaja. El caso interesante es
Elastic Net, con dos ($\alpha$ y `l1_ratio`) — el escenario donde `06-seleccion-hiperparametros.md`
predice que random search y la optimización bayesiana empiezan a tener ventaja. Conviene
comprobar si con este dataset la ventaja llega a verse, en vez de darla por hecha.

In [ ]:
pipe_en = crear_pipeline(ElasticNet(max_iter=20000))

# Grid: 8 x 8 = 64 combinaciones evaluadas todas.
grid_en = {"reg__alpha": np.logspace(-3, 1, 8), "reg__l1_ratio": np.linspace(0.05, 0.95, 8)}
t0 = time.time()
gs_en = GridSearchCV(pipe_en, grid_en, cv=kf, scoring="neg_root_mean_squared_error")
gs_en.fit(X_train, y_train)
t_grid = time.time() - t0

# Random: 25 combinaciones muestreadas de una distribución continua.
dist_en = {"reg__alpha": loguniform(1e-3, 10), "reg__l1_ratio": uniform(0.05, 0.9)}
t0 = time.time()
rs_en = RandomizedSearchCV(
    pipe_en, dist_en, n_iter=25, cv=kf, scoring="neg_root_mean_squared_error", random_state=SEMILLA
)
rs_en.fit(X_train, y_train)
t_random = time.time() - t0

resultados_busqueda = pd.DataFrame(
    [
        {"método": "Grid (64 combos)", "evaluaciones": 64, "tiempo_s": t_grid, "RMSE_cv": -gs_en.best_score_},
        {"método": "Random (25 iter)", "evaluaciones": 25, "tiempo_s": t_random, "RMSE_cv": -rs_en.best_score_},
    ]
)
resultados_busqueda.round(2)

### Optuna: reutiliza el historial para decidir dónde probar

La función objetivo hace su propia validación cruzada con el mismo `kf`, para que la
comparación con grid y random search sea justa.

> **Los primeros trials de TPE son aleatorios.** `TPESampler` no empieza a usar el historial
> de inmediato: necesita algunas evaluaciones para construir su modelo probabilístico, y
> hasta entonces muestrea al azar. El valor por defecto de `n_startup_trials` es **10**, así
> que una corrida de 10 trials con la configuración de fábrica sería **random search puro** —
> TPE no llegaría a intervenir ni una vez. Aquí se baja a 5 para que, incluso en la corrida
> más corta, la mitad de las evaluaciones sí usen el historial. Es justo el tipo de detalle
> que decide si un experimento mide lo que uno cree que mide.

In [ ]:
def objetivo(trial):
    alpha = trial.suggest_float("alpha", 1e-3, 10, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0.05, 0.95)
    modelo = crear_pipeline(ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=20000))
    rmse_por_pliegue = []
    for tr_idx, va_idx in kf.split(X_train):
        modelo.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        pred = modelo.predict(X_train.iloc[va_idx])
        rmse_por_pliegue.append(mean_squared_error(y_train.iloc[va_idx], pred) ** 0.5)
    return float(np.mean(rmse_por_pliegue))


resultados_optuna = []
for n_trials in [10, 25]:
    t0 = time.time()
    # seed: sección 4 de 06-seleccion-hiperparametros.md.
    # n_startup_trials=5: con el valor por defecto (10), la corrida de 10 trials no usaría TPE.
    sampler = optuna.samplers.TPESampler(seed=SEMILLA, n_startup_trials=5)
    estudio = optuna.create_study(direction="minimize", sampler=sampler)
    estudio.optimize(objetivo, n_trials=n_trials, show_progress_bar=False)
    resultados_optuna.append(
        {
            "método": f"Optuna ({n_trials} trials)",
            "evaluaciones": n_trials,
            "tiempo_s": time.time() - t0,
            "RMSE_cv": estudio.best_value,
        }
    )

pd.concat([resultados_busqueda, pd.DataFrame(resultados_optuna)], ignore_index=True).round(2)

Lo primero que hay que notar es lo que la tabla **no** dice. Los cuatro RMSE caben en un
rango de \$4 sobre un error de \$34,377 — un 0.01 %. Sobre este problema las cuatro
búsquedas encontraron esencialmente el mismo óptimo, y leer "Optuna con 25 trials ganó por
seis centavos" como que un método le gana a otro sería justo el error que la sección 5
enseña a no cometer. **Una diferencia más chica que su propia incertidumbre no es un
resultado.**

La comparación informativa es la otra columna: **cuánto cuesta llegar ahí**. Grid gastó 64
evaluaciones para el mismo sitio al que random llegó con 25, y Optuna con 25 — y casi con
10. Con dos hiperparámetros y una superficie tan plana, la rejilla exhaustiva no compra
nada: simplemente paga más.

Ningún método "hace trampa": los cuatro ajustan y validan exactamente igual, y la diferencia
está en cómo deciden qué combinación probar después. Pero conviene ser explícito sobre lo
que este experimento **no** demuestra: con un espacio de búsqueda tan benigno, la ventaja de
la optimización bayesiana no tiene dónde manifestarse. Para verla de verdad hacen falta más
dimensiones y evaluaciones caras — el escenario que describe
`06-seleccion-hiperparametros.md` y que reaparece en el módulo 4 con los ensambles.

## 4. ¿Cuánto optimismo hay en reportar el error sobre el mismo CV que eligió $\lambda$?

`06-seleccion-hiperparametros.md`, sección 3: elegir el hiperparámetro y reportar el error
de ese mismo proceso de selección es optimista. Se mide comparando el RMSE "ingenuo" de la
sección 2 contra una CV anidada de verdad.

In [ ]:
rmse_ingenuo = -busqueda_ridge.best_score_  # de la seccion 2: mismo CV elige y reporta

errores_externos = []
for tr_idx, va_idx in KFold(n_splits=5, shuffle=True, random_state=SEMILLA).split(X_train):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

    cv_interno = KFold(n_splits=5, shuffle=True, random_state=SEMILLA)
    busqueda_interna = GridSearchCV(pipe_ridge, grid_ridge, cv=cv_interno, scoring="neg_root_mean_squared_error")
    busqueda_interna.fit(X_tr, y_tr)  # elige alpha SOLO con datos de este pliegue externo

    pred_externa = busqueda_interna.predict(X_va)  # evalúa en el pliegue que el interno nunca vio
    errores_externos.append(mean_squared_error(y_va, pred_externa) ** 0.5)

rmse_anidado = np.mean(errores_externos)
ee_anidado = np.std(errores_externos) / np.sqrt(len(errores_externos))

print(f"RMSE ingenuo (mismo CV elige y reporta): ${rmse_ingenuo:,.0f}")
print(f"RMSE de CV anidada:                      ${rmse_anidado:,.0f} ± ${ee_anidado:,.0f}")
print(f"Optimismo medido:                        ${rmse_anidado - rmse_ingenuo:,.0f}")

El optimismo medido es pequeño frente al error estándar de la propia estimación anidada —en
este dataset, con $\lambda$ óptimo ya cercano a OLS, no hay mucho margen de fuga—. No
siempre es así: cuantos más hiperparámetros y más pequeño el dataset, mayor la brecha
esperada entre el número "ingenuo" y el real. La CV anidada es la manera de **saber** cuál es
el caso, en vez de asumirlo.

## 5. Comparación pareada: ¿Ridge le gana a Lasso, de verdad?

Se afinan ambos con grid search (cada uno con su propio $\lambda$ óptimo) y se comparan
sobre los **mismos** 10 pliegues — la comparación pareada de `05-sesgo-varianza-validacion.md`.

In [ ]:
REJILLA_LAMBDA = {"reg__alpha": np.logspace(-2, 3, 20)}

busqueda_lasso = GridSearchCV(
    crear_pipeline(Lasso(max_iter=20000)),
    REJILLA_LAMBDA,
    cv=kf,
    scoring="neg_root_mean_squared_error",
)
busqueda_lasso.fit(X_train, y_train)

alpha_ridge = busqueda_ridge.best_params_["reg__alpha"]
alpha_lasso = busqueda_lasso.best_params_["reg__alpha"]

modelo_ridge = crear_pipeline(Ridge(alpha=alpha_ridge))
modelo_lasso = crear_pipeline(Lasso(alpha=alpha_lasso, max_iter=20000))

kf_pareado = KFold(n_splits=10, shuffle=True, random_state=SEMILLA)
errores_ridge, errores_lasso = [], []
for tr_idx, va_idx in kf_pareado.split(X_train):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
    modelo_ridge.fit(X_tr, y_tr)
    modelo_lasso.fit(X_tr, y_tr)
    errores_ridge.append(mean_squared_error(y_va, modelo_ridge.predict(X_va)) ** 0.5)
    errores_lasso.append(mean_squared_error(y_va, modelo_lasso.predict(X_va)) ** 0.5)


def resumir_diferencias(err_ridge, err_lasso, etiqueta):
    """Diferencia pliegue a pliegue, su error estándar, y el cociente entre ambos."""
    d = np.array(err_ridge) - np.array(err_lasso)
    ee = d.std() / np.sqrt(len(d))
    print(etiqueta)
    print(f"  RMSE medio — Ridge: ${np.mean(err_ridge):,.0f}   Lasso: ${np.mean(err_lasso):,.0f}")
    print(f"  Diferencia media (Ridge − Lasso): ${d.mean():+,.1f}")
    print(f"  Error estándar de la diferencia:  ${ee:,.1f}")
    print(f"  |diferencia| / ee = {abs(d.mean()) / ee:.2f}")
    return d.mean(), ee


print(f"Ridge (α={alpha_ridge:.1f}) vs. Lasso (α={alpha_lasso:.1f})\n")
dif_fuga, ee_fuga = resumir_diferencias(
    errores_ridge, errores_lasso, "(a) λ fijo, elegido de antemano con todo X_train:"
)

La diferencia media es mucho menor que su propio error estándar, así que la lectura sería
"no hay evidencia de que uno le gane al otro". **Pero este experimento tiene exactamente el
defecto que la sección 4 acaba de medir.** Los dos $\lambda$ se eligieron con `kf` sobre
**todo** `X_train`, y después se evalúan sobre 10 pliegues de ese mismo `X_train`: cada
pliegue de validación ya había participado en elegir ambos hiperparámetros. Es la fuga de la
sección 3 de `06-seleccion-hiperparametros.md`, ahora metida dentro de una comparación.

La versión correcta usa la misma idea de la CV anidada: que **cada pliegue elija su propio
$\lambda$** con datos que no lo incluyen. Así se compara lo que de verdad interesa —el
procedimiento completo "afinar Ridge" contra "afinar Lasso"— y no dos modelos cuyo
hiperparámetro ya vio los datos con los que se los evalúa.

In [ ]:
errores_ridge_anid, errores_lasso_anid = [], []
for tr_idx, va_idx in kf_pareado.split(X_train):  # los MISMOS 10 pliegues de antes
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
    cv_interno = KFold(n_splits=5, shuffle=True, random_state=SEMILLA)

    ridge_afinado = GridSearchCV(
        crear_pipeline(Ridge()), REJILLA_LAMBDA, cv=cv_interno,
        scoring="neg_root_mean_squared_error",
    ).fit(X_tr, y_tr)
    lasso_afinado = GridSearchCV(
        crear_pipeline(Lasso(max_iter=20000)), REJILLA_LAMBDA, cv=cv_interno,
        scoring="neg_root_mean_squared_error",
    ).fit(X_tr, y_tr)

    errores_ridge_anid.append(mean_squared_error(y_va, ridge_afinado.predict(X_va)) ** 0.5)
    errores_lasso_anid.append(mean_squared_error(y_va, lasso_afinado.predict(X_va)) ** 0.5)

dif_limpia, ee_limpia = resumir_diferencias(
    errores_ridge_anid, errores_lasso_anid, "(b) λ elegido dentro de cada pliegue:"
)

Corregir la fuga no solo movió el número — **cambió la conclusión**:

| Comparación | Dif. media (Ridge − Lasso) | ee | \|dif\| / ee |
|---|---|---|---|
| (a) $\lambda$ fijo, elegido con todo `X_train` | −\$10 | \$35 | 0.29 |
| (b) $\lambda$ elegido dentro de cada pliegue | −\$41 | \$17 | 2.36 |

La fuga metía ruido que ensanchaba el error estándar —\$35 contra \$17— y ahogaba una
diferencia que sí estaba ahí. Sin ella, Ridge le gana a Lasso por unos \$41, algo por encima
de la regla práctica de dos errores estándar.

**Y aun así, la decisión práctica no cambia.** Dos advertencias, en orden:

1. El error estándar de un k-fold **subestima** la incertidumbre real, porque los pliegues
   comparten la mayoría de sus datos de entrenamiento (`05-sesgo-varianza-validacion.md`,
   sección 2). Un cociente de 2.36 está en el borde de la regla, no holgadamente por encima.
2. Más importante: \$41 sobre un RMSE de \$33,460 es un **0.12 %**. "Estadísticamente
   detectable" y "relevante para decidir" no son lo mismo, y esta diferencia es lo primero
   sin ser lo segundo. Nadie elige entre Ridge y Lasso por 41 dólares; se elige por
   interpretabilidad, por costo de cómputo, o por si conviene que el modelo anule variables.

Compárese con `ej03-validacion-cruzada.md`, que aplica esta misma herramienta a otra pregunta
—¿aporta algo la variable `foundation`?— y encuentra \$961 con un ee de \$211: cociente 4.6
**y** una magnitud que sí mueve una decisión. Esa es la diferencia entre un hallazgo y una
curiosidad estadística.

## 6. El número final: el conjunto de prueba

Todo lo anterior ocurrió dentro de `X_train`. Queda el paso que cierra cualquier proyecto
honesto: entrenar el modelo elegido sobre **todo** el entrenamiento y medirlo **una sola
vez** sobre el 20 % que no participó en nada — ni en ajustar, ni en elegir $\lambda$, ni en
comparar modelos.

In [ ]:
modelo_final = crear_pipeline(Ridge(alpha=alpha_ridge))
modelo_final.fit(X_train, y_train)

rmse_test = mean_squared_error(y_test, modelo_final.predict(X_test)) ** 0.5
rmse_base_test = mean_squared_error(y_test, np.full(len(y_test), y_train.mean())) ** 0.5

resumen_final = pd.DataFrame(
    {
        "estimación": [
            "Línea base (predecir el promedio)",
            "CV ingenua, 5 pliegues (sección 2)",
            "CV anidada (sección 4)",
            "Conjunto de prueba (una sola vez)",
        ],
        "RMSE": [rmse_base_test, rmse_ingenuo, rmse_anidado, rmse_test],
    }
)
resumen_final.round(0)

Los tres números del modelo quedan muy por debajo de la línea base, pero no son idénticos
entre sí, y vale la pena mirar por qué. El RMSE de prueba (\$37,966) queda unos \$3,500 por
encima de las estimaciones de CV (\$34,377 y \$34,450) — dentro de un error estándar de la
CV anidada (\$3,569), pero justo en el límite. No es una contradicción: miden cosas
distintas sobre conjuntos distintos, y este 20 % en particular resultó algo más difícil que
el promedio de los pliegues. Es, de paso, una demostración más de la lección de la sesión 7:
**un solo conjunto es una muestra, y tiene su propia variabilidad**. Si el número de prueba
hubiera caído \$3,500 por debajo en vez de por encima, la tentación de celebrarlo habría
sido igual de infundada.

Lo que sí distingue a ese número de los otros tres: es el único que se puede reportar como
desempeño esperado sin matices — y solo se puede usar **una vez**. Si ahora se volviera
atrás a probar otro modelo y se reportara el mejor de los dos sobre `X_test`, `X_test`
habría dejado de ser un conjunto de prueba y volveríamos al problema de la sección 4.

## Resumen del módulo 3

| Sesión | Lo que se resolvió |
|---|---|
| S6 | Regresión lineal y descenso del gradiente, validados contra la solución exacta |
| S7 | Multicolinealidad, VIF, Ridge/Lasso a mano, y su efecto real (o no) sobre la predicción |
| S8 | Sesgo-varianza, k-fold, grid/random/Optuna, CV anidada, comparación pareada de modelos |

Las tres promesas pendientes de los módulos 1 y 2 quedan cumplidas: la variabilidad de una
sola partición se mide con CV en vez de esconderse (sección 2), la incertidumbre de una
métrica y de la diferencia entre modelos se cuantifica en vez de asumirse (secciones 4 y 5),
y la búsqueda de hiperparámetros ya no se hace a ojo con un solo split de validación.

Y una lección que no estaba en el plan original de la sesión: la comparación de la sección 5
**cambia de conclusión** según se respete o no la separación entre elegir el hiperparámetro y
evaluar. La higiene metodológica no es un trámite que se hace al final para quedar bien — es
lo que decide qué se puede afirmar.